# Lecture 30: Ant Colony Optimization - Motivation & Pseudocode

---

```{note}
Simulated Annealing (Lectures 24-26) refines one solution; the Genetic Algorithm (Lectures 27-29) evolves a population through selection and recombination. **Ant Colony Optimization (ACO)**, Lecture 23's third paradigm, takes a different approach again: a colony of simple agents ("ants") repeatedly *construct* candidate solutions, coordinating with each other only indirectly, through a shared trail of pheromone. Unlike the population and swarm methods common to other courses, ACO was built from the ground up for exactly the kind of combinatorial routing problem this module has used throughout — no encoding trick required.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Explain how indirect coordination through pheromone (stigmergy) lets a colony of simple agents solve a combinatorial problem with no central controller.
2. Read and interpret Ant Colony Optimization's pseudocode, including the construction rule and the evaporation/deposit update.
3. Hand-trace one ant's tour construction and the resulting pheromone update for a small instance.

**Prerequisites**: Genetic Algorithm - Benchmarking (Lecture 29); Metaheuristics (Lecture 23).

**Estimated time**: 50 minutes

---

## Why Ant Colony Optimization?

A real ant colony finds short paths between its nest and a food source with no map, no leader, and no ant that can see the whole trail network. Each ant deposits a chemical **pheromone** as it walks, and is more likely to follow a path already carrying more pheromone. Shorter paths get walked — and so reinforced — more often per unit time than longer ones, so pheromone accumulates faster on them; over many ants and many trips, the colony's paths converge on something close to the shortest route, purely through this indirect, trail-mediated feedback. This mechanism — agents coordinating not by talking to each other, but by modifying and reacting to a shared environment — is called **stigmergy**.

**Ant Colony Optimization** turns this into a search algorithm. Each iteration, a colony of $m$ artificial ants each **constructs** a complete candidate solution — for a routing problem, a full tour — one step at a time, choosing the next stop probabilistically, biased toward edges with more pheromone and toward edges that are simply short (a built-in heuristic preference, since real ants also tend to prefer nearby food). Once every ant has finished, pheromone **evaporates** everywhere a little, and every ant **deposits** fresh pheromone along the tour it just built, in an amount inversely proportional to that tour's length — shorter tours reinforce their edges more strongly. Repeated over many iterations, this reinforce-what-worked, fade-what-didn't cycle concentrates the colony's construction choices onto increasingly good tours.

```{note}
Compare this to Simulated Annealing and the Genetic Algorithm: SA refines one solution by moving through a neighbourhood; GA evolves a population by recombining existing solutions. ACO does neither — it **builds a new solution from scratch every iteration**, one decision at a time, guided by a shared memory (pheromone) that persists and evolves across iterations. Because that construction process is inherently a sequence of discrete choices, ACO fits a routing problem's permutation structure directly — unlike a swarm intelligence algorithm built for continuous spaces, which would need an encoding bridge to touch a permutation at all.
```

---

## Notation

| Symbol | Meaning |
|--------|---------|
| $\tau_{ij}$ | Pheromone level on edge $(i,j)$ |
| $\eta_{ij} = 1/d_{ij}$ | Heuristic desirability ("visibility") of edge $(i,j)$ — higher for shorter edges |
| $\alpha$ | Pheromone importance weight |
| $\beta$ | Heuristic importance weight |
| $\rho$ | Evaporation rate ($0 < \rho < 1$) |
| $Q$ | Pheromone deposit constant |
| $m$ | Number of ants |
| $U$ | The set of stops a given ant has not yet visited |

**Construction rule**: an ant currently at stop $i$, with unvisited stops $U$, chooses the next stop $j \in U$ with probability

$$p_{ij} = \frac{\tau_{ij}^\alpha\, \eta_{ij}^\beta}{\displaystyle\sum_{l \in U} \tau_{il}^\alpha\, \eta_{il}^\beta}$$

```{tip}
$\alpha$ and $\beta$ play the same exploration/exploitation role as every other parameter this module has calibrated: $\alpha=0$ ignores pheromone entirely (the ant always prefers the nearest unvisited stop, a greedy nearest-neighbour construction); $\beta=0$ ignores distance entirely (the ant follows pheromone alone, however the colony happened to lay it). Setting both above zero blends "what has worked for the colony so far" with "what looks locally promising right now" — exactly the same blend SA's acceptance rule and the Genetic Algorithm's fitness-weighted selection are built from, expressed here as a single probability instead of an accept/reject test or a selection rule.
```

---

## Pseudocode

1. **Procedure** $\text{ACO}(\text{graph}, \alpha, \beta, \rho, Q, m)$
2. $\tau_{ij} \leftarrow \tau_o$ for every edge $(i,j)$ &emsp;<small>// initialise pheromone uniformly</small>
3. **while** $!\text{converged}$ **do**
4. &emsp;**for** ant $k = 1, \ldots, m$ **do**
5. &emsp;&emsp;construct a full tour $s_k$: starting from the depot, repeatedly choose the next stop from $U$ using the construction rule above, until $U = \emptyset$, then return to the depot
6. &emsp;**end for**
7. &emsp;$\tau_{ij} \leftarrow (1-\rho)\,\tau_{ij}$ for every edge $(i,j)$ &emsp;<small>// evaporate</small>
8. &emsp;**for** ant $k = 1, \ldots, m$ **do**
9. &emsp;&emsp;**for** every edge $(i,j)$ in $s_k$ **do** $\tau_{ij} \leftarrow \tau_{ij} + Q / f(s_k)$ &emsp;<small>// deposit, proportional to tour quality</small>
10. &emsp;**end for**
11. &emsp;$s^* \leftarrow$ the best of $s_1, \ldots, s_m$, if better than the incumbent
12. **end while**
13. **return** $s^*$

```{caution}
Notice there is no explicit "current solution" or "population" carried between iterations the way $s$ or $\boldsymbol{s}$ were in Lectures 24 and 27 — the only thing that persists from one iteration to the next is the pheromone matrix $\tau$. Every ant constructs its tour completely fresh each iteration; it is the *pheromone*, not any individual ant, that accumulates the colony's learning over time.
```

---

## Applying ACO to Continuous Problems

Every part of ACO's construction rule assumes a **graph**: discrete nodes, discrete edges, a pheromone value living on each edge. None of that exists for a continuous function like the Ackley function Lecture 31 will need for its first hands-on implementation — there is no finite edge set to deposit pheromone on. This is the mirror image of the encoding problem a continuous-space algorithm would face on a routing problem.

```{note}
**ACOR** (Ant Colony Optimization for continuous domains; Socha & Dorigo, 2008) resolves this by replacing the pheromone matrix with an **archive** of the $k$ best real-valued solutions found so far, each carrying a weight based on its rank (better solutions weighted more heavily). A new candidate is built dimension-by-dimension: for each dimension, one archive member is picked at random (favouring higher-weighted, better-ranked members), and a new value is sampled from a Gaussian distribution centred on that member's value, with a spread based on how close together the archive's members already are in that dimension. New candidates are merged into the archive, which is then trimmed back to its $k$ best — playing the same reinforce-what-worked, fade-what-didn't role that evaporation and deposit play for pheromone on a graph.
```

Lecture 31 implements this in full — and its job ends there: once the module starts working with real routing benchmarks (Lecture 32 onward), those benchmarks *are* graphs, and native ACO applies with no bridge needed at all. For now, the key idea is: **the archive replaces the pheromone matrix as "what the colony remembers,"** and Gaussian sampling around archive members replaces graph-edge construction as "how a new candidate gets built."

---

## Hand-Traced Example

Consider a 4-stop instance — a depot (node 0) and three stops (1, 2, 3) — with distances $d_{01}=2$, $d_{02}=3$, $d_{03}=4$, $d_{12}=3$, $d_{13}=5$, $d_{23}=2$. Initialize $\tau_{ij}=1$ on every edge, and trace **one ant's construction**, starting at the depot, with $\alpha=1$, $\beta=2$.

**Step 1 — at the depot, $U=\{1,2,3\}$.** Since all $\tau_{ij}=1$, the construction probabilities depend only on $\eta_{ij}^2 = (1/d_{0j})^2$:

$$p_{01} = \frac{2^{-2}}{2^{-2}+3^{-2}+4^{-2}} = 0.590 \qquad p_{02} = 0.262 \qquad p_{03} = 0.148$$

Given a random draw $r=0.5$: cumulative probability reaches $0.590$ at stop 1, and $0.5 < 0.590$, so **stop 1 is chosen**.

**Step 2 — at stop 1, $U=\{2,3\}$.** $\tau$ is still uniform (it only updates after *all* ants finish their tours, not mid-construction):

$$p_{12} = \frac{3^{-2}}{3^{-2}+5^{-2}} = 0.735 \qquad p_{13} = 0.265$$

With $r=0.5$: cumulative probability reaches $0.735$ at stop 2, and $0.5 < 0.735$, so **stop 2 is chosen**.

**Step 3 — at stop 2, $U=\{3\}$.** Only one choice remains: stop 3, then return to the depot.

**Tour**: $0 \to 1 \to 2 \to 3 \to 0$, length $= 2+3+2+4 = 11$.

**Pheromone update** ($\rho=0.5$, $Q=10$): every edge first evaporates, $\tau_{ij} \leftarrow 0.5 \times 1 = 0.5$; then this ant deposits $Q/11 \approx 0.909$ on each edge of its tour ($0\text{-}1$, $1\text{-}2$, $2\text{-}3$, $3\text{-}0$), bringing those edges to $\tau \approx 1.409$, while edges the ant never used (e.g. $0\text{-}2$, $1\text{-}3$) are left at $\tau=0.5$ — evaporated, with nothing to replace what faded.

In a single pass, the edges this one ant happened to use are now noticeably more attractive to the *next* ant's construction than the edges it didn't — exactly the reinforcement Lecture 23's exploration/exploitation framing predicts, here emerging from nothing but evaporation and proportional deposit.

---

## In-Class Exercise

### Exercise 1 — A Second Ant, Same Iteration

Continuing this lecture's example, before any evaporation or deposit happens, a second ant also constructs a tour from the depot, using the *original* uniform pheromone ($\tau_{ij}=1$ throughout, since pheromone updates only after every ant in the iteration has finished), but drawing $r=0.9$ at the depot step instead of $0.5$.

At the depot, the cumulative probabilities from this lecture's Step 1 are $0.590$ (stop 1), $0.590+0.262=0.852$ (stop 2), $0.852+0.148=1.0$ (stop 3). Since $r=0.9 > 0.852$, **stop 3 is chosen** this time — a less likely, but not impossible, outcome exactly because $p_{03}=0.148$ was small but nonzero.

Continue this ant's construction: at stop 3, $U=\{1,2\}$, with $p_{31} \propto 5^{-2}=0.04$ and $p_{32} \propto 2^{-2}=0.25$, giving $p_{31}=0.138$, $p_{32}=0.862$. A further draw of $r=0.5$ falls under the cumulative probability for stop 2 ($0.862$), so stop 2 is chosen next, leaving stop 1 last. This ant's tour: $0\to3\to2\to1\to0$, length $=4+2+3+2=11$ — the same length as the first ant's tour, reached by the reverse-ish route. **Once both ants' tours are known, evaporation and deposit are applied once, together** (not once per ant) — with two ants both depositing on their (different, in general) tour edges, an edge used by both ants receives two deposits.

---

## Take-Away Exercises

### Exercise 1 — A Poor Random Draw

Redo this lecture's main hand-trace, but with the depot-step draw $r=0.99$ instead of $0.5$ (choosing stop 3 first, the least-preferred option), followed by whatever draws you choose for the remaining steps. Compute the resulting tour length and compare it to the $11$ this lecture's example found. Does a single "unlucky" ant meaningfully harm the colony's pheromone trails after just one iteration, given that $m$ ants (not just one) deposit before the next iteration's ants construct anything?

### Exercise 2 — Heuristic-Only Construction

Set $\alpha=0$ (ignore pheromone entirely) and repeat the depot-step probability calculation. Confirm that the construction becomes a pure "prefer the nearest unvisited stop, weighted by $\eta^\beta$" rule, independent of any pheromone history. What does this predict about how ACO behaves in its very first iteration, before any ant has deposited anything?

---

## Circling Back

- **Lecture 23 (Metaheuristics)**: Ant Colony Optimization is this module's example of the *swarm intelligence* paradigm — but coordinated through a shared external memory (pheromone) rather than through each agent tracking its own and the group's best-known position.
- **Lecture 27 (Genetic Algorithm: Motivation & Pseudocode)**: both GA and ACO maintain something that persists and accumulates improvement across iterations — a population there, a pheromone matrix here — but GA's population *is* a set of candidate solutions, while ACO's pheromone is not a solution at all, only a bias on how future solutions get built.

## Moving Forward

- **Lecture 31 (Ant Colony Optimization: Algorithm)**: implements ACOR — the continuous-domain bridge introduced above — in Python, and applies it to the same Ackley function Lectures 25 and 28 already used. ACOR's role ends there; Lecture 32 moves to real TSPLIB routing benchmarks, where ACO's native graph-based form (this lecture's own pseudocode) applies directly.

---

## Further Reading

- Dorigo, M. (1992). *Optimization, Learning and Natural Algorithms*. PhD thesis, Politecnico di Milano — the origin of Ant System, ACO's first instantiation.
- Dorigo, M., Maniezzo, V., and Colorni, A. (1996). "Ant System: Optimization by a Colony of Cooperating Agents." *IEEE Transactions on Systems, Man, and Cybernetics, Part B*, 26(1), 29-41.
- Dorigo, M. and Stützle, T. (2004). *Ant Colony Optimization*. MIT Press — the standard reference text.
- Socha, K. and Dorigo, M. (2008). "Ant Colony Optimization for Continuous Domains." *European Journal of Operational Research*, 185(3), 1155-1173 — the ACOR extension previewed above, developed in Lecture 31.